<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/12_GES_Aware_Genomic_RAG_Cell_7C5_LLM_Generation_Execution_Authorization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm that Google Drive is mounted and the project folder is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, frozen identities, exact hashes, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import csv
import hashlib
import json
import re
import tempfile

import numpy as np
import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = '12_GES_Aware_Genomic_RAG_Cell_7C5_LLM_Generation_Execution_Authorization.ipynb'
CELL_ID = '7C5'
STAGE = '7C'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_PROMPTS = 480
EXPECTED_QUESTIONS = 80
EXPECTED_ALIASES = 6
EXPECTED_REPETITIONS = 3
EXPECTED_RUN_IDS = [0, 1, 2]
EXPECTED_GENERATION_REQUESTS = 1_440

EXPECTED_CELL_7C4_TERMINAL_DECISION = (
    'PASS_STAGE7C4_480_SCORE_BLIND_PROMPTS_AND_2400_CONTEXT_BLOCKS_MATERIALIZED_'
    'FROM_EXACT_CELL7C2_TOP5_CELL7B3_SCORE_BLIND_CORPUS_AND_QUESTIONS_USING_EXACT_'
    'CELL7B4_FROZEN_TEMPLATES_CHECKSUM_PROTECTED_NO_SCORE_BEARING_AUDIT_CELL7A3_SCORES_'
    'GES_QUALITY_RANK_SEMANTIC_RANK_RRF_UNBLINDED_CONDITION_IDENTITY_ANSWER_KEYS_'
    'LLM_RESPONSES_ADJUDICATION_OR_RAG_METRICS_NEXT_EXECUTION_NOT_AUTHORIZED'
)

EXPECTED_CELL_7B4_TERMINAL_DECISION = (
    'PASS_STAGE7B4_EXACT_EMBEDDING_MODEL_REVISION_TEXT_NORMALIZATION_'
    'SIMILARITY_TOP20_CANDIDATE_POOL_TOP5_CONTEXT_QUALITY_RERANKING_'
    'BLINDED_ALIASES_PROMPTS_STRICT_RESPONSE_SCHEMA_FIXED_LLM_SNAPSHOT_'
    'GENERATION_RUNTIME_AND_DETERMINISTIC_CONTROLS_FROZEN_CHECKSUM_PROTECTED_'
    'NO_EMBEDDINGS_RETRIEVAL_RERANKING_PROMPT_MATERIALIZATION_LLM_'
    'ANSWER_KEY_OUTCOME_INSPECTION_OR_RAG_EVALUATION_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------------------
# Successful Cell 7C4 package.
# --------------------------------------------------------------------------------------------------
CELL_7C4_EXEC_DIR = (
    ROOT / 'outputs' / 'rag_execution' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)
CELL_7C4_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)
CELL_7C4_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c4_score_blind_prompt_materialization_v1'
)

CELL_7C4 = OrderedDict([
    ('prompt_instances', {
        'path': CELL_7C4_EXEC_DIR / 'cell_7c4_score_blind_prompt_instances_v1.parquet',
        'sha256': 'd5b088815d7710fce5c79b89b39828787a745d360404d20270319e91fdcd4ce5',
    }),
    ('context_inventory', {
        'path': CELL_7C4_EXEC_DIR / 'cell_7c4_score_blind_prompt_context_inventory_v1.parquet',
        'sha256': 'd23b604dac645c158cd1b95cc9cd564fb6b9447279756b6cae589b5572dd4623',
    }),
    ('input_inventory', {
        'path': CELL_7C4_CONFIG_DIR / 'cell_7c4_verified_prompt_input_inventory_v1.csv',
        'sha256': 'd7af53e5767fef07f5175928033c0176bb9863377ee50cc885dc55a7ba42362c',
    }),
    ('execution_report', {
        'path': CELL_7C4_QC_DIR / 'cell_7c4_prompt_materialization_execution_report_v1.json',
        'sha256': '0bf99fd699b9addf25b24a1a9945d19b2a7e9fc826c04f5a204cbd9bba63ede6',
    }),
    ('qc', {
        'path': CELL_7C4_QC_DIR / 'cell_7c4_prompt_materialization_qc_v1.json',
        'sha256': '9130769728e0b99c566abed3cecadcb8322797ac88b5c21d0a036eca787a977a',
    }),
    ('manifest', {
        'path': CELL_7C4_CONFIG_DIR / 'cell_7c4_prompt_materialization_manifest_v1.json',
        'sha256': 'a58c4b61e2c603a496047d5b36174d466e59cf293491115b364c6466480ba31d',
    }),
])

# --------------------------------------------------------------------------------------------------
# Exact Cell 7B4 generation freeze.
# --------------------------------------------------------------------------------------------------
CELL_7B4_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7b4_configuration_freeze_v1'
)
CELL_7B4_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7b4_configuration_freeze_v1'
)

CELL_7B4 = OrderedDict([
    ('llm_prompt_response', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_llm_prompt_response_configuration_v1.json',
        'sha256': 'e3f684f9c471b8074dc41f2398f8aff0cb03217f188e2eab2ad8f4c0070a810b',
    }),
    ('runtime_determinism', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_runtime_and_determinism_configuration_v1.json',
        'sha256': '6003c85ef151ae1d7dca530462fa1dbcf6983be4b7e92744b8e65c8d8b42b1d3',
    }),
    ('condition_aliases', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_condition_alias_inventory_v1.csv',
        'sha256': '6eb45683b42a456d2b6788a5fcf6b9cd95fc606afe9627610ebbc11914312cb9',
    }),
    ('requirements_lock', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_execution_requirements_lock_v1.txt',
        'sha256': '1a894f7ba976d00563325cc3324ff91708b5699e1c618e3674502fe586828a91',
    }),
    ('qc', {
        'path': CELL_7B4_QC_DIR / 'cell_7b4_configuration_freeze_qc_v1.json',
        'sha256': '01d6ef836c25d49b9df32d2739d1d532a1f205be0d83c6227eb65070d9f9ced4',
    }),
    ('manifest', {
        'path': CELL_7B4_CONFIG_DIR / 'cell_7b4_configuration_freeze_manifest_v1.json',
        'sha256': '18a5d6cb3cadca0eab950839a19022686fc6bad2c398ed87f2a48c66eff462fe',
    }),
])

# --------------------------------------------------------------------------------------------------
# Cell 7C5 authorization outputs.
# --------------------------------------------------------------------------------------------------
AUTH_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c5_llm_generation_authorization_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c5_llm_generation_authorization_v1'
)

OUTPUTS = OrderedDict([
    ('authorization',
     AUTH_DIR / 'cell_7c5_stage7c_cell7c6_llm_generation_authorization_v1.json'),
    ('generation_plan',
     AUTH_DIR / 'cell_7c5_frozen_generation_request_plan_v1.parquet'),
    ('input_inventory',
     AUTH_DIR / 'cell_7c5_authorized_generation_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'cell_7c5_llm_generation_authorization_qc_v1.json'),
    ('manifest',
     AUTH_DIR / 'cell_7c5_llm_generation_authorization_manifest_v1.json'),
])

for directory in (AUTH_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if OUTPUTS['manifest'].exists():
    raise FileExistsError(
        f'Cell 7C5 manifest already exists: {OUTPUTS["manifest"]}\\n'
        'Fail-closed overwrite protection is active.'
    )

print(f'Authorization directory: {AUTH_DIR}')
print(f'QC directory           : {QC_DIR}')

Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c5_llm_generation_authorization_v1
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c5_llm_generation_authorization_v1


## 2. Strict checksum, sidecar, JSON/CSV/Parquet, and serialization helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(
            f'Invalid SHA-256 sidecar format: {path}\\n'
            f'Observed first token: {token!r}'
        )
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    sc = sidecar_path(path)
    return (
        path.exists()
        and sc.exists()
        and read_sidecar_hash(sc) == sha256_file(path)
    )


def verify_exact_artifact(label: str, path: Path, expected_sha256: str) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'SHA-256 mismatch for {label}.\\n'
            f'Expected: {expected_sha256}\\n'
            f'Observed: {observed}\\n'
            f'Path: {path}'
        )
    if not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing SHA-256 sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)),
        'sidecar_valid': True,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def parquet_metadata(path: Path) -> dict[str, Any]:
    pf = pq.ParquetFile(path)
    return {
        'rows': int(pf.metadata.num_rows),
        'columns': int(pf.metadata.num_columns),
        'schema_names': list(pf.schema_arrow.names),
    }


def to_json_native(value: Any) -> Any:
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, np.generic):
        return to_json_native(value.item())
    if isinstance(value, np.ndarray):
        return [to_json_native(v) for v in value.tolist()]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): to_json_native(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [to_json_native(v) for v in value]
    if hasattr(value, 'item'):
        return to_json_native(value.item())
    raise TypeError(f'Unsupported JSON type: {type(value).__name__}')


def stable_write_json(path: Path, payload: Any) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    native = to_json_native(payload)
    text = json.dumps(
        native,
        indent=2,
        sort_keys=True,
        ensure_ascii=False,
        allow_nan=False,
    ) + chr(10)
    path.write_text(text, encoding='utf-8')
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(
        path,
        index=False,
        encoding='utf-8',
        lineterminator=chr(10),
    )
    return sha256_file(path)


def stable_write_parquet(path: Path, frame: pd.DataFrame) -> str:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_parquet(
        path,
        index=False,
        engine='pyarrow',
        compression='zstd',
    )
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    digest = sha256_file(path)
    sidecar_path(path).write_text(
        f'{digest}  {path.name}' + chr(10),
        encoding='utf-8',
    )


# Writer self-test. This catches both prior failure modes before touching Cell 7C5 outputs.
with tempfile.TemporaryDirectory(prefix='cell_7c5_writer_selftest_') as _tmp:
    _tmpdir = Path(_tmp)

    _json_path = _tmpdir / 'test.json'
    _payload = {'ok': True, 'nested': {'value': 7}}
    stable_write_json(_json_path, _payload)
    if json.loads(_json_path.read_text(encoding='utf-8')) != _payload:
        raise AssertionError('JSON writer round-trip self-test failed.')
    if not _json_path.read_bytes().endswith(bytes([10])):
        raise AssertionError('JSON writer did not emit a real LF byte.')

    _csv_path = _tmpdir / 'test.csv'
    stable_write_csv(_csv_path, pd.DataFrame([{'a': 1}, {'a': 2}]))
    if bytes([10]) not in _csv_path.read_bytes():
        raise AssertionError('CSV writer did not emit real LF bytes.')

    write_sidecar(_json_path)
    if not sidecar_is_valid(_json_path):
        raise AssertionError('SHA-256 sidecar writer/parser self-test failed.')

print('Serialization and SHA-256 helper self-test: PASS')

Serialization and SHA-256 helper self-test: PASS


## 3. Reverify the complete successful Cell 7C4 prompt package

In [4]:
verified_inputs: list[dict[str, Any]] = []
verified_7c4 = OrderedDict()

for artifact_id, spec in CELL_7C4.items():
    record = verify_exact_artifact(
        f'cell_7c4_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7C4'
    verified_7c4[artifact_id] = record
    verified_inputs.append(record)

report_7c4 = load_json(CELL_7C4['execution_report']['path'])
qc_7c4 = load_json(CELL_7C4['qc']['path'])
manifest_7c4 = load_json(CELL_7C4['manifest']['path'])

if manifest_7c4.get('terminal_decision') != EXPECTED_CELL_7C4_TERMINAL_DECISION:
    raise AssertionError(
        'Cell 7C4 terminal decision mismatch.\\n'
        f'Observed: {manifest_7c4.get("terminal_decision")}'
    )
if manifest_7c4.get('next_authorized_cell') is not None:
    raise AssertionError('Cell 7C4 unexpectedly authorizes a downstream cell.')
if manifest_7c4.get('llm_execution_authorized') is not False:
    raise AssertionError('Cell 7C4 must not authorize LLM execution.')
if int(qc_7c4.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C4 QC does not report zero failures.')
if report_7c4.get('scientific_operations', {}).get('llm_called') is not False:
    raise AssertionError('Cell 7C4 execution report indicates an LLM call.')

prompt_meta = parquet_metadata(CELL_7C4['prompt_instances']['path'])
context_meta = parquet_metadata(CELL_7C4['context_inventory']['path'])

EXPECTED_PROMPT_SCHEMA = [
    'prompt_instance_id',
    'question_id',
    'blinded_alias',
    'question_text_sha256',
    'context_count',
    'context_packet_ids_json',
    'context_rcv_accessions_json',
    'context_block_hashes_json',
    'context_bundle_sha256',
    'system_prompt',
    'system_prompt_sha256',
    'user_prompt',
    'user_prompt_sha256',
    'full_prompt_sha256',
]

if prompt_meta['rows'] != EXPECTED_PROMPTS:
    raise AssertionError(f'Cell 7C4 prompt row count changed: {prompt_meta["rows"]}')
if prompt_meta['schema_names'] != EXPECTED_PROMPT_SCHEMA:
    raise AssertionError(
        'Cell 7C4 prompt schema changed.\\n'
        f'Observed: {prompt_meta["schema_names"]}'
    )
if context_meta['rows'] != 2_400:
    raise AssertionError(f'Cell 7C4 context row count changed: {context_meta["rows"]}')

# Open the score-blind prompt package only.
prompts = pd.read_parquet(CELL_7C4['prompt_instances']['path'])

for column in ('prompt_instance_id', 'question_id', 'blinded_alias', 'system_prompt', 'user_prompt'):
    prompts[column] = prompts[column].astype('string')

if len(prompts) != EXPECTED_PROMPTS:
    raise AssertionError('Loaded Cell 7C4 prompt row count changed.')
if prompts['prompt_instance_id'].isna().any() or prompts['prompt_instance_id'].duplicated().any():
    raise AssertionError('Prompt-instance IDs are missing or duplicated.')
if prompts[['question_id', 'blinded_alias']].duplicated().any():
    raise AssertionError('Duplicate question/blinded-alias prompt found.')
if prompts['question_id'].nunique(dropna=False) != EXPECTED_QUESTIONS:
    raise AssertionError('Prompt package does not contain exactly 80 questions.')
if prompts['blinded_alias'].nunique(dropna=False) != EXPECTED_ALIASES:
    raise AssertionError('Prompt package does not contain exactly six blinded aliases.')
if not prompts['context_count'].eq(5).all():
    raise AssertionError('Every frozen prompt must contain exactly five contexts.')
if prompts['system_prompt'].str.strip().eq('').any():
    raise AssertionError('Blank system prompt found.')
if prompts['user_prompt'].str.strip().eq('').any():
    raise AssertionError('Blank user prompt found.')

# Recompute prompt hashes from the frozen text.
if not all(
    sha256_text(str(text)) == str(digest)
    for text, digest in zip(prompts['system_prompt'], prompts['system_prompt_sha256'])
):
    raise AssertionError('At least one system-prompt hash does not reproduce.')

if not all(
    sha256_text(str(text)) == str(digest)
    for text, digest in zip(prompts['user_prompt'], prompts['user_prompt_sha256'])
):
    raise AssertionError('At least one user-prompt hash does not reproduce.')

# Score/condition leakage must remain absent from the prompt artifact schema.
PROHIBITED_PROMPT_COLUMN_TOKENS = (
    'condition_id',
    'condition_name',
    'condition_role',
    'ges',
    'p_stable',
    'instability',
    'quality_signal',
    'quality_rank',
    'semantic_rank',
    'semantic_score',
    'rrf',
    'random_quality',
)
leaking_prompt_columns = [
    column
    for column in prompts.columns
    if any(token in column.lower() for token in PROHIBITED_PROMPT_COLUMN_TOKENS)
]
if leaking_prompt_columns:
    raise AssertionError(
        'Score/condition leakage columns found in Cell 7C4 prompt package: '
        + ', '.join(leaking_prompt_columns)
    )

print('Cell 7C4 complete frozen package          : 6/6 exact hashes + sidecars')
print('Cell 7C4 terminal PASS                   : VERIFIED')
print(f'Frozen prompt instances                  : {len(prompts):,}')
print(f'Frozen context rows                      : {context_meta["rows"]:,}')
print('Score-bearing Cell 7C2 audit loaded      : NO')
print('Cell 7A3 scores loaded                   : NO')
print('Answer-key outcomes inspected            : NO')
print('LLM called in Cell 7C5                  : NO')

Cell 7C4 complete frozen package          : 6/6 exact hashes + sidecars
Cell 7C4 terminal PASS                   : VERIFIED
Frozen prompt instances                  : 480
Frozen context rows                      : 2,400
Score-bearing Cell 7C2 audit loaded      : NO
Cell 7A3 scores loaded                   : NO
Answer-key outcomes inspected            : NO
LLM called in Cell 7C5                  : NO


## 4. Reverify the exact Cell 7B4 LLM snapshot, response schema, generation settings, and runtime freeze

In [5]:
verified_7b4 = OrderedDict()

for artifact_id, spec in CELL_7B4.items():
    record = verify_exact_artifact(
        f'cell_7b4_{artifact_id}',
        spec['path'],
        spec['sha256'],
    )
    record['source_cell'] = '7B4'
    verified_7b4[artifact_id] = record
    verified_inputs.append(record)

llm_config = load_json(CELL_7B4['llm_prompt_response']['path'])
runtime_config = load_json(CELL_7B4['runtime_determinism']['path'])
qc_7b4 = load_json(CELL_7B4['qc']['path'])
manifest_7b4 = load_json(CELL_7B4['manifest']['path'])

if manifest_7b4.get('terminal_decision') != EXPECTED_CELL_7B4_TERMINAL_DECISION:
    raise AssertionError('Cell 7B4 terminal PASS decision mismatch.')
if int(qc_7b4.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7B4 QC does not report zero failures.')

llm = llm_config.get('llm', {})
generation = llm_config.get('generation', {})
prompt_cfg = llm_config.get('prompt', {})
structured = llm_config.get('structured_output', {})
response_schema = structured.get('schema', {})

EXPECTED_RESPONSE_FIELDS = {
    'answer',
    'clinical_significance',
    'conflict_detected',
    'evidence_strength',
    'response_policy',
    'confidence',
    'evidence_ids',
    'reasoning_summary',
}

generation_checks = OrderedDict([
    ('provider_openai', llm.get('provider') == 'OpenAI'),
    ('api_responses', llm.get('api') == 'Responses API'),
    ('model_snapshot_exact', llm.get('model') == 'gpt-4.1-mini-2025-04-14'),
    ('fixed_snapshot_required', llm.get('fixed_snapshot_required') is True),
    ('tools_empty', llm.get('tools') == []),
    ('web_search_false', llm.get('web_search') is False),
    ('file_search_false', llm.get('file_search') is False),
    ('code_interpreter_false', llm.get('code_interpreter') is False),
    ('store_false', llm.get('store') is False),
    ('stream_false', llm.get('stream') is False),
    ('temperature_zero', generation.get('temperature') == 0.0),
    ('top_p_one', generation.get('top_p') == 1.0),
    ('max_output_tokens_1200', generation.get('max_output_tokens') == 1200),
    ('presence_penalty_zero', generation.get('presence_penalty') == 0.0),
    ('frequency_penalty_zero', generation.get('frequency_penalty') == 0.0),
    ('three_repetitions', generation.get('repetitions_per_question_condition') == 3),
    ('run_ids_0_1_2', generation.get('run_ids') == EXPECTED_RUN_IDS),
    ('score_values_not_in_prompt', prompt_cfg.get('score_values_in_prompt') is False),
    ('condition_identity_not_in_prompt', prompt_cfg.get('condition_identity_in_prompt') is False),
    ('answer_key_not_in_prompt', prompt_cfg.get('answer_key_in_prompt') is False),
    ('strict_structured_output', structured.get('strict') is True),
    ('json_schema_output', structured.get('type') == 'json_schema'),
    ('response_schema_no_additional_properties',
     response_schema.get('additionalProperties') is False),
    ('response_schema_required_exact',
     set(response_schema.get('required', [])) == EXPECTED_RESPONSE_FIELDS),
    ('response_schema_properties_exact',
     set(response_schema.get('properties', {}).keys()) == EXPECTED_RESPONSE_FIELDS),
])

failed_generation_checks = [
    name for name, passed in generation_checks.items()
    if not bool(passed)
]
if failed_generation_checks:
    raise RuntimeError(
        'Frozen Cell 7B4 generation configuration validation failed:\\n- '
        + '\\n- '.join(failed_generation_checks)
    )

# The exact prompt-system text embedded in Cell 7C4 must still match Cell 7B4.
expected_system_hash = prompt_cfg.get('system_prompt_sha256')
if not prompts['system_prompt_sha256'].astype(str).eq(str(expected_system_hash)).all():
    raise AssertionError('Cell 7C4 system-prompt hash does not match Cell 7B4.')

expected_alias_order = [
    'ARM-MICA',
    'ARM-ORBIT',
    'ARM-KITE',
    'ARM-PULSE',
    'ARM-LARCH',
    'ARM-NOVA',
]
with CELL_7B4['condition_aliases']['path'].open('r', encoding='utf-8', newline='') as handle:
    alias_rows = list(csv.DictReader(handle))

observed_alias_order = [row['blinded_alias'] for row in alias_rows]
if observed_alias_order != expected_alias_order:
    raise AssertionError(f'Frozen alias order changed: {observed_alias_order}')
if set(prompts['blinded_alias'].astype(str)) != set(expected_alias_order):
    raise AssertionError('Cell 7C4 prompt aliases do not match Cell 7B4 frozen aliases.')

print('Cell 7B4 LLM/runtime package              : 6/6 exact hashes + sidecars')
print('Cell 7B4 terminal PASS                   : VERIFIED')
print('Model snapshot                           : gpt-4.1-mini-2025-04-14')
print('API                                      : Responses API')
print('Temperature / top-p                     : 0.0 / 1.0')
print('Max output tokens                        : 1200')
print('Repetitions / run IDs                   : 3 / [0, 1, 2]')
print('Strict response schema                   : VERIFIED')
print('Tools / web / file search / code interp  : DISABLED')
print('Answer-key access                        : NOT AUTHORIZED')

Cell 7B4 LLM/runtime package              : 6/6 exact hashes + sidecars
Cell 7B4 terminal PASS                   : VERIFIED
Model snapshot                           : gpt-4.1-mini-2025-04-14
API                                      : Responses API
Temperature / top-p                     : 0.0 / 1.0
Max output tokens                        : 1200
Repetitions / run IDs                   : 3 / [0, 1, 2]
Strict response schema                   : VERIFIED
Tools / web / file search / code interp  : DISABLED
Answer-key access                        : NOT AUTHORIZED


## 5. Freeze the exact 1,440-request generation plan

In [6]:
# The authorization plan carries hashes and identities only, not answer keys or score-bearing data.
alias_order_map = {alias: i for i, alias in enumerate(expected_alias_order)}

prompts_for_plan = prompts[
    [
        'prompt_instance_id',
        'question_id',
        'blinded_alias',
        'context_count',
        'system_prompt_sha256',
        'user_prompt_sha256',
        'full_prompt_sha256',
    ]
].copy()

prompts_for_plan['_alias_order'] = (
    prompts_for_plan['blinded_alias']
    .map(alias_order_map)
    .astype('int16')
)

plan_parts = []
for run_id in EXPECTED_RUN_IDS:
    part = prompts_for_plan.copy()
    part['run_id'] = np.int16(run_id)
    plan_parts.append(part)

generation_plan = pd.concat(plan_parts, ignore_index=True)
generation_plan = generation_plan.sort_values(
    ['run_id', 'question_id', '_alias_order'],
    kind='mergesort',
).drop(columns=['_alias_order']).reset_index(drop=True)

generation_plan.insert(
    0,
    'generation_request_id',
    [
        f'{prompt_instance_id}__run{int(run_id)}'
        for prompt_instance_id, run_id in zip(
            generation_plan['prompt_instance_id'],
            generation_plan['run_id'],
        )
    ],
)

generation_plan['model_snapshot'] = llm['model']
generation_plan['api'] = llm['api']
generation_plan['temperature'] = float(generation['temperature'])
generation_plan['top_p'] = float(generation['top_p'])
generation_plan['max_output_tokens'] = int(generation['max_output_tokens'])
generation_plan['presence_penalty'] = float(generation['presence_penalty'])
generation_plan['frequency_penalty'] = float(generation['frequency_penalty'])
generation_plan['strict_json_schema'] = bool(structured['strict'])
generation_plan['response_schema_name'] = str(structured.get('name', 'clinical_genomic_evidence_answer_v1'))
generation_plan['response_schema_sha256'] = sha256_text(
    json.dumps(
        response_schema,
        ensure_ascii=False,
        sort_keys=True,
        separators=(',', ':'),
    )
)
generation_plan['tools_enabled'] = False
generation_plan['web_search_enabled'] = False
generation_plan['file_search_enabled'] = False
generation_plan['code_interpreter_enabled'] = False
generation_plan['answer_key_access_authorized'] = False
generation_plan['evaluation_authorized'] = False

# Deterministic plan QC.
plan_checks = OrderedDict([
    ('generation_requests_1440', len(generation_plan) == EXPECTED_GENERATION_REQUESTS),
    ('unique_request_ids', not generation_plan['generation_request_id'].duplicated().any()),
    ('each_prompt_three_requests',
     generation_plan.groupby('prompt_instance_id').size().eq(3).all()),
    ('run_ids_exact',
     set(generation_plan['run_id'].astype(int).tolist()) == set(EXPECTED_RUN_IDS)),
    ('each_run_480_requests',
     generation_plan.groupby('run_id').size().eq(480).all()),
    ('80_questions_each_run',
     generation_plan.groupby('run_id')['question_id'].nunique().eq(80).all()),
    ('6_aliases_each_run',
     generation_plan.groupby('run_id')['blinded_alias'].nunique().eq(6).all()),
    ('context_count_always_5', generation_plan['context_count'].eq(5).all()),
    ('model_snapshot_exact',
     generation_plan['model_snapshot'].eq('gpt-4.1-mini-2025-04-14').all()),
    ('responses_api_exact', generation_plan['api'].eq('Responses API').all()),
    ('temperature_zero', generation_plan['temperature'].eq(0.0).all()),
    ('top_p_one', generation_plan['top_p'].eq(1.0).all()),
    ('max_output_tokens_1200', generation_plan['max_output_tokens'].eq(1200).all()),
    ('strict_schema_true', generation_plan['strict_json_schema'].eq(True).all()),
    ('no_tools', generation_plan['tools_enabled'].eq(False).all()),
    ('no_web_search', generation_plan['web_search_enabled'].eq(False).all()),
    ('no_file_search', generation_plan['file_search_enabled'].eq(False).all()),
    ('no_code_interpreter', generation_plan['code_interpreter_enabled'].eq(False).all()),
    ('answer_key_access_false',
     generation_plan['answer_key_access_authorized'].eq(False).all()),
    ('evaluation_authorized_false',
     generation_plan['evaluation_authorized'].eq(False).all()),
])

failed_plan_checks = [
    name for name, passed in plan_checks.items()
    if not bool(passed)
]
if failed_plan_checks:
    raise RuntimeError(
        'Frozen generation-plan QC failed:\\n- '
        + '\\n- '.join(failed_plan_checks)
    )

print(f'Frozen prompt instances                  : {len(prompts):,}')
print(f'Frozen repetitions per prompt            : {EXPECTED_REPETITIONS}')
print(f'Frozen generation requests               : {len(generation_plan):,}')
print('Unique generation-request IDs            : YES')
print('Prompt modification                      : NO')
print('Answer-key access                        : NO')
print('Evaluation                               : NO')
print('LLM calls executed in Cell 7C5           : NO')

Frozen prompt instances                  : 480
Frozen repetitions per prompt            : 3
Frozen generation requests               : 1,440
Unique generation-request IDs            : YES
Prompt modification                      : NO
Answer-key access                        : NO
Evaluation                               : NO
LLM calls executed in Cell 7C5           : NO


## 6. Freeze Cell 7C6 authorization, generation plan, QC, manifest, and immutable readback

In [7]:
authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C6_EXACT_1440_FROZEN_LLM_GENERATION_REQUESTS_'
    '480_SCORE_BLIND_PROMPTS_X3_RUNS_FIXED_GPT41MINI_20250414_RESPONSES_API_'
    'TEMPERATURE0_TOPP1_MAXTOKENS1200_STRICT_JSON_SCHEMA_NO_TOOLS_WEB_FILESEARCH_'
    'CODEINTERPRETER_PROMPT_MODIFICATION_ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)

AUTHORIZED_CELL = {
    'stage': '7C',
    'cell_id': '7C6',
    'title': 'Exact frozen score-blind LLM response generation',
    'authorized_once': True,
    'overwrite_existing_frozen_outputs': False,
}

AUTHORIZED_OPERATIONS = [
    'Load the exact checksum-frozen Cell 7C4 score-blind prompt package.',
    'Load the exact checksum-frozen Cell 7C5 generation request plan.',
    'Execute exactly the 1,440 planned request identities: 480 prompt instances x run IDs 0, 1, 2.',
    'Use only the frozen model snapshot gpt-4.1-mini-2025-04-14 via the Responses API.',
    'Use the exact frozen generation settings and strict JSON response schema from Cell 7B4.',
    'Disable tools, web search, file search, code interpreter, streaming, and response storage exactly as frozen.',
    'Freeze request/response provenance, raw response payloads needed for audit, parsed structured outputs, execution status, and SHA-256 sidecars.',
    'Permit restart/resume only by generation_request_id without changing the frozen request content or configuration.',
]

PROHIBITED_OPERATIONS = [
    'Load the Cell 7C2 score-bearing reranking audit.',
    'Load Cell 7A3 score rows.',
    'Load or inspect structured answer-key outcomes.',
    'Modify system prompts, user prompts, evidence context membership, context order, or prompt hashes.',
    'Substitute a different model, model alias, API, temperature, top-p, token limit, response schema, or generation setting.',
    'Enable tools, web search, file search, code interpreter, or streaming.',
    'Use response content to change remaining prompts or generation settings.',
    'Perform adjudication.',
    'Calculate retrieval, answer, RAG, bootstrap, or statistical performance metrics.',
]

# Input inventory.
input_records = []
for artifact_id, record in verified_7c4.items():
    copied = dict(record)
    copied['source_cell'] = '7C4'
    copied['row_level_content_loaded_in_cell_7c5'] = artifact_id == 'prompt_instances'
    input_records.append(copied)

for artifact_id, record in verified_7b4.items():
    copied = dict(record)
    copied['source_cell'] = '7B4'
    copied['row_level_content_loaded_in_cell_7c5'] = artifact_id in {
        'llm_prompt_response',
        'runtime_determinism',
        'condition_aliases',
    }
    input_records.append(copied)

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization_type': 'fail_closed_scientific_execution_authorization',
    'authorization_decision': authorization_decision,
    'authorized_cell': AUTHORIZED_CELL,
    'authorized_operations': AUTHORIZED_OPERATIONS,
    'prohibited_operations': PROHIBITED_OPERATIONS,
    'frozen_generation_design': {
        'prompt_instances': EXPECTED_PROMPTS,
        'questions': EXPECTED_QUESTIONS,
        'blinded_aliases': EXPECTED_ALIASES,
        'repetitions_per_prompt': EXPECTED_REPETITIONS,
        'run_ids': EXPECTED_RUN_IDS,
        'generation_requests': EXPECTED_GENERATION_REQUESTS,
        'model_snapshot': llm['model'],
        'api': llm['api'],
        'temperature': generation['temperature'],
        'top_p': generation['top_p'],
        'max_output_tokens': generation['max_output_tokens'],
        'presence_penalty': generation['presence_penalty'],
        'frequency_penalty': generation['frequency_penalty'],
        'strict_json_schema': structured['strict'],
        'tools': llm['tools'],
        'web_search': llm['web_search'],
        'file_search': llm['file_search'],
        'code_interpreter': llm['code_interpreter'],
        'store': llm['store'],
        'stream': llm['stream'],
    },
    'answer_key_access_authorized': False,
    'evaluation_authorized': False,
    'adjudication_authorized': False,
    'cell_7c2_score_bearing_audit_loaded': False,
    'cell_7a3_scores_loaded': False,
    'llm_called_in_cell_7c5': False,
}

# Write outputs using real LF bytes.
stable_write_json(OUTPUTS['authorization'], authorization_payload)
write_sidecar(OUTPUTS['authorization'])

stable_write_parquet(OUTPUTS['generation_plan'], generation_plan)
write_sidecar(OUTPUTS['generation_plan'])

input_inventory = pd.DataFrame(input_records)
stable_write_csv(OUTPUTS['input_inventory'], input_inventory)
write_sidecar(OUTPUTS['input_inventory'])

checks = OrderedDict([
    ('cell_7c4_complete_package_6_verified', len(verified_7c4) == 6),
    ('cell_7c4_all_sidecars_valid', all(v['sidecar_valid'] for v in verified_7c4.values())),
    ('cell_7c4_terminal_pass_exact',
     manifest_7c4['terminal_decision'] == EXPECTED_CELL_7C4_TERMINAL_DECISION),
    ('cell_7c4_qc_zero_failures', int(qc_7c4['failed_checks']) == 0),
    ('cell_7c4_prompt_rows_480', len(prompts) == 480),
    ('cell_7c4_context_rows_2400', context_meta['rows'] == 2400),
    ('cell_7c4_no_leaking_prompt_columns', len(leaking_prompt_columns) == 0),
    ('cell_7b4_complete_package_6_verified', len(verified_7b4) == 6),
    ('cell_7b4_all_sidecars_valid', all(v['sidecar_valid'] for v in verified_7b4.values())),
    ('cell_7b4_terminal_pass_exact',
     manifest_7b4['terminal_decision'] == EXPECTED_CELL_7B4_TERMINAL_DECISION),
    ('all_generation_config_checks_pass', all(bool(v) for v in generation_checks.values())),
    ('all_generation_plan_checks_pass', all(bool(v) for v in plan_checks.values())),
    ('generation_plan_rows_1440', len(generation_plan) == 1440),
    ('authorization_targets_cell_7c6', AUTHORIZED_CELL['cell_id'] == '7C6'),
    ('authorization_once_true', AUTHORIZED_CELL['authorized_once'] is True),
    ('answer_key_access_false', authorization_payload['answer_key_access_authorized'] is False),
    ('evaluation_authorized_false', authorization_payload['evaluation_authorized'] is False),
    ('adjudication_authorized_false', authorization_payload['adjudication_authorized'] is False),
    ('score_bearing_audit_not_loaded',
     authorization_payload['cell_7c2_score_bearing_audit_loaded'] is False),
    ('cell7a3_scores_not_loaded', authorization_payload['cell_7a3_scores_loaded'] is False),
    ('llm_not_called_in_7c5', authorization_payload['llm_called_in_cell_7c5'] is False),
])

failed_checks = [
    name for name, passed in checks.items()
    if not bool(passed)
]
if failed_checks:
    raise RuntimeError(
        'Cell 7C5 authorization QC failed:\\n- '
        + '\\n- '.join(failed_checks)
    )

terminal_decision = (
    'PASS_STAGE7C5_COMPLETE_CELL7C4_480_SCORE_BLIND_PROMPTS_AND_CELL7B4_FIXED_LLM_RUNTIME_'
    'CONFIG_REVERIFIED_1440_REQUEST_GENERATION_PLAN_FROZEN_CHECKSUM_PROTECTED_CELL7C6_'
    'EXACT_LLM_GENERATION_ONLY_AUTHORIZED_NO_SCORE_BEARING_AUDIT_CELL7A3_SCORES_'
    'ANSWER_KEYS_ADJUDICATION_OR_RAG_METRICS'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'generation_config_checks': {
        name: bool(value) for name, value in generation_checks.items()
    },
    'generation_plan_checks': {
        name: bool(value) for name, value in plan_checks.items()
    },
    'authorization_checks': {
        name: bool(value) for name, value in checks.items()
    },
    'passed_checks': int(
        len(generation_checks) + len(plan_checks) + len(checks)
    ),
    'failed_checks': 0,
    'total_checks': int(
        len(generation_checks) + len(plan_checks) + len(checks)
    ),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'project_root': str(ROOT),
    'authorization_decision': authorization_decision,
    'authorization_lineage': {
        'cell_7c4_prompt_instances_sha256': CELL_7C4['prompt_instances']['sha256'],
        'cell_7c4_manifest_sha256': CELL_7C4['manifest']['sha256'],
        'cell_7b4_llm_prompt_response_sha256': CELL_7B4['llm_prompt_response']['sha256'],
        'cell_7b4_runtime_determinism_sha256': CELL_7B4['runtime_determinism']['sha256'],
        'cell_7b4_manifest_sha256': CELL_7B4['manifest']['sha256'],
    },
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_path': str(sidecar_path(path)),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C6',
    'answer_key_access_authorized': False,
    'evaluation_authorized': False,
    'next_required_action': (
        'Execute Cell 7C6 only for the exact 1,440 frozen score-blind LLM generation requests. '
        'A separate authorization is required afterward before answer-key access or evaluation.'
    ),
}
stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

# Fresh immutable readback.
for key, path in OUTPUTS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing Cell 7C5 output: {path}')
    if not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C5 output sidecar failed: {path}')

auth_readback = load_json(OUTPUTS['authorization'])
plan_readback = pd.read_parquet(OUTPUTS['generation_plan'])
qc_readback = load_json(OUTPUTS['qc'])
manifest_readback = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('authorization_decision_exact',
     auth_readback['authorization_decision'] == authorization_decision),
    ('authorized_cell_7c6',
     auth_readback['authorized_cell']['cell_id'] == '7C6'),
    ('generation_plan_1440', len(plan_readback) == 1440),
    ('generation_plan_unique_request_ids',
     not plan_readback['generation_request_id'].duplicated().any()),
    ('generation_plan_each_prompt_three',
     plan_readback.groupby('prompt_instance_id').size().eq(3).all()),
    ('qc_zero_failures', int(qc_readback['failed_checks']) == 0),
    ('manifest_terminal_exact',
     manifest_readback['terminal_decision'] == terminal_decision),
    ('manifest_next_cell_7c6',
     manifest_readback['next_authorized_cell'] == '7C6'),
    ('manifest_answer_key_false',
     manifest_readback['answer_key_access_authorized'] is False),
    ('manifest_evaluation_false',
     manifest_readback['evaluation_authorized'] is False),
    ('all_five_output_sidecars_valid',
     all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_readback = [
    name for name, passed in readback_checks.items()
    if not bool(passed)
]
if failed_readback:
    raise RuntimeError(
        'Cell 7C5 final readback QC failed:\\n- '
        + '\\n- '.join(failed_readback)
    )

passed_checks = (
    len(generation_checks)
    + len(plan_checks)
    + len(checks)
    + len(readback_checks)
)
total_checks = passed_checks

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C5')
print('FROZEN LLM-GENERATION EXECUTION AUTHORIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nUPSTREAM CELL 7C4 REVERIFICATION')
print(f'Cell 7C4 manifest SHA-256                     : {sha256_file(CELL_7C4["manifest"]["path"])}')
print('Cell 7C4 terminal PASS verified               : YES')
print('Frozen Cell 7C4 artifacts                     : 6/6 exact hashes + sidecars')
print(f'Score-blind prompt instances                  : {len(prompts):,}')
print(f'Score-blind context rows                      : {context_meta["rows"]:,}')
print('Score-bearing Cell 7C2 audit loaded           : NO')
print('Cell 7A3 scores loaded                        : NO')
print('Answer-key outcomes inspected                 : NO')

print('\\nFROZEN GENERATION CONFIGURATION')
print(f'Model snapshot                                : {llm["model"]}')
print(f'API                                           : {llm["api"]}')
print(f'Temperature                                   : {generation["temperature"]}')
print(f'Top-p                                         : {generation["top_p"]}')
print(f'Max output tokens                             : {generation["max_output_tokens"]}')
print(f'Repetitions                                   : {generation["repetitions_per_question_condition"]}')
print(f'Run IDs                                       : {generation["run_ids"]}')
print('Strict JSON response schema                   : VERIFIED')
print('Tools / web / file search / code interpreter  : DISABLED')

print('\\nCELL 7C6 AUTHORIZATION')
print(f'Frozen prompt instances                       : {EXPECTED_PROMPTS}')
print(f'Repetitions per prompt                        : {EXPECTED_REPETITIONS}')
print(f'Frozen generation requests                    : {len(plan_readback):,}')
print('Prompt modification                           : PROHIBITED')
print('Model/settings substitution                   : PROHIBITED')
print('Answer-key access                             : PROHIBITED')
print('Adjudication / RAG metrics                    : PROHIBITED')

print('\\nCELL 7C5 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {passed_checks}/{total_checks} PASS')

print('\\nSCIENTIFIC OPERATIONS IN CELL 7C5')
print('Generation request plan frozen                : YES')
print('LLM called                                   : NO')
print('Responses generated                          : NO')
print('Answer-key outcomes inspected                : NO')
print('Adjudication or RAG metrics                  : NO')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C6                           : exact frozen LLM generation only')
print('Planned API calls                             : 1,440')
print('Answer-key access / evaluation                : PROHIBITED')
print('Post-generation evaluation                    : requires separate authorization')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C5
FROZEN LLM-GENERATION EXECUTION AUTHORIZATION
Notebook                                      : 12_GES_Aware_Genomic_RAG_Cell_7C5_LLM_Generation_Execution_Authorization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nUPSTREAM CELL 7C4 REVERIFICATION
Cell 7C4 manifest SHA-256                     : a58c4b61e2c603a496047d5b36174d466e59cf293491115b364c6466480ba31d
Cell 7C4 terminal PASS verified               : YES
Frozen Cell 7C4 artifacts                     : 6/6 exact hashes + sidecars
Score-blind prompt instances                  : 480
Score-blind context rows                      : 2,400
Score-bearing Cell 7C2 audit loaded           : NO
Cell 7A3 scores loaded                        : NO
Answer-key outcomes inspected                 : NO
\n